# Week 10: Capstone 完整示例

## 学习目标

1. 综合应用 ML 和 OR 方法解决实际问题
2. 完成完整的项目流程
3. 培养系统性思维
4. 展示项目成果

## 项目背景

**问题**：智能单车共享系统优化

**目标**：
1. 预测各站点需求
2. 优化车辆调度
3. 评估站点布局
4. 提供决策支持

## 1. 环境准备

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pulp import *

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Capstone 项目工具已加载")

## 2. 数据准备

In [ ]:
# 生成模拟数据
def generate_bike_data(n_days=90):
    """生成单车共享系统模拟数据"""
    
    stations = ['A001', 'A002', 'A003', 'B001', 'B002', 'C001']
    station_types = {
        'A001': '商业区', 'A002': '住宅区', 'A003': '商业区',
        'B001': '住宅区', 'B002': '混合区', 'C001': '景区'
    }
    
    data = []
    
    for day in range(n_days):
        is_weekend = day % 7 >= 5
        is_holiday = day in [15, 45, 75]  # 模拟节假日
        
        for hour in range(24):
            for station in stations:
                # 基础需求
                base_demand = {
                    'A001': 25, 'A002': 15, 'A003': 20,
                    'B001': 12, 'B002': 18, 'C001': 22
                }[station]
                
                # 时间因素
                hour_factor = 1.0
                if 7 <= hour <= 9:  # 早高峰
                    hour_factor = 1.8
                elif 17 <= hour <= 19:  # 晚高峰
                    hour_factor = 1.6
                elif 0 <= hour <= 5:  # 夜间
                    hour_factor = 0.3
                
                # 周末效应
                weekend_factor = 0.6 if is_weekend else 1.0
                
                # 节假日效应
                holiday_factor = 1.3 if is_holiday else 1.0
                
                # 站点类型效应
                type_factor = {
                    '商业区': 1.2 if not is_weekend else 0.8,
                    '住宅区': 0.9 if not is_weekend else 1.1,
                    '混合区': 1.0,
                    '景区': 0.7 if not is_weekend else 1.4
                }[station_types[station]]
                
                # 温度（模拟）
                temperature = 15 + 10 * np.sin(2 * np.pi * day / 90) + np.random.normal(0, 3)
                temp_factor = 1 + 0.02 * (temperature - 20)
                
                # 计算需求
                demand = base_demand * hour_factor * weekend_factor * \
                        holiday_factor * type_factor * temp_factor
                demand = max(0, demand + np.random.normal(0, 3))
                
                data.append({
                    'day': day,
                    'hour': hour,
                    'station_id': station,
                    'station_type': station_types[station],
                    'is_weekend': is_weekend,
                    'is_holiday': is_holiday,
                    'temperature': temperature,
                    'demand': demand
                })
    
    return pd.DataFrame(data)

# 生成数据
df = generate_bike_data(90)

print(f"数据形状: {df.shape}")
print(f"\n数据概览:")
print(df.head())

## 3. 探索性数据分析

In [ ]:
# 需求统计
print("需求统计")
print("=" * 60)
print(df.groupby('station_id')['demand'].agg(['mean', 'std', 'min', 'max']))

In [ ]:
# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 各站点需求分布
df.boxplot(column='demand', by='station_id', ax=axes[0, 0])
axes[0, 0].set_title('各站点需求分布')
plt.suptitle('')

# 2. 24 小时需求曲线
hourly_demand = df.groupby('hour')['demand'].mean()
axes[0, 1].plot(hourly_demand.index, hourly_demand.values, marker='o')
axes[0, 1].set_xlabel('小时')
axes[0, 1].set_ylabel('平均需求')
axes[0, 1].set_title('24 小时需求曲线')
axes[0, 1].grid(True, alpha=0.3)

# 3. 工作日 vs 周末
weekend_compare = df.groupby(['is_weekend', 'hour'])['demand'].mean().unstack(level=0)
axes[1, 0].plot(weekend_compare.index, weekend_compare[0], label='工作日', marker='o')
axes[1, 0].plot(weekend_compare.index, weekend_compare[1], label='周末', marker='s')
axes[1, 0].set_xlabel('小时')
axes[1, 0].set_ylabel('平均需求')
axes[1, 0].set_title('工作日 vs 周末需求')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 温度 vs 需求
axes[1, 1].scatter(df['temperature'], df['demand'], alpha=0.3)
axes[1, 1].set_xlabel('温度 (°C)')
axes[1, 1].set_ylabel('需求')
axes[1, 1].set_title('温度 vs 需求')

plt.tight_layout()
plt.show()

## 4. 需求预测模型

In [ ]:
# 特征工程
df['is_rush_hour'] = ((df['hour'] >= 7) & (df['hour'] <= 9) | 
                      (df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)

df['is_night'] = (df['hour'] <= 5).astype(int)

# 站点编码
station_encoder = {s: i for i, s in enumerate(df['station_id'].unique())}
df['station_encoded'] = df['station_id'].map(station_encoder)

# 特征和目标
feature_cols = ['hour', 'is_weekend', 'is_holiday', 'temperature', 
                'is_rush_hour', 'is_night', 'station_encoded']

X = df[feature_cols]
y = df['demand']

# 分割数据
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")

In [ ]:
# 模型训练
models = {
    '线性回归': LinearRegression(),
    '随机森林': RandomForestRegressor(n_estimators=100, random_state=42),
    '梯度提升': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        '模型': name,
        'R²': r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })

results_df = pd.DataFrame(results)
print("需求预测模型性能")
print("=" * 60)
print(results_df.to_string(index=False))

In [ ]:
# 选择最佳模型
best_model = models['随机森林']

# 特征重要性
importance = pd.DataFrame({
    '特征': feature_cols,
    '重要性': best_model.feature_importances_
}).sort_values('重要性', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 特征重要性
axes[0].barh(importance['特征'], importance['重要性'])
axes[0].set_xlabel('重要性')
axes[0].set_title('特征重要性')

# 预测 vs 实际
y_pred = best_model.predict(X_test)
axes[1].scatter(y_test, y_pred, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('实际需求')
axes[1].set_ylabel('预测需求')
axes[1].set_title('预测 vs 实际')

plt.tight_layout()
plt.show()

## 5. 车辆调度优化

In [ ]:
# 使用预测模型生成调度需求
# 模拟早高峰时段的站点需求

stations = df['station_id'].unique()
current_bikes = {s: 20 for s in stations}  # 假设各站点当前有 20 辆车

# 预测早高峰需求
morning_peak_features = []
for station in stations:
    morning_peak_features.append({
        'hour': 8,
        'is_weekend': 0,
        'is_holiday': 0,
        'temperature': 20,
        'is_rush_hour': 1,
        'is_night': 0,
        'station_encoded': station_encoder[station]
    })

morning_df = pd.DataFrame(morning_peak_features)
predicted_demand = best_model.predict(morning_df)

target_bikes = {s: max(5, int(d)) for s, d in zip(stations, predicted_demand)}

print("早高峰时段站点需求")
print("=" * 60)
for s in stations:
    print(f"{s}: 当前 {current_bikes[s]} 辆 -> 目标 {target_bikes[s]} 辆")

In [ ]:
# 车辆调度优化模型
def optimize_rebalancing(stations, current, target, cost_matrix):
    """
    优化车辆调度
    
    参数:
    - stations: 站点列表
    - current: 当前车辆数
    - target: 目标车辆数
    - cost_matrix: 站点间移动成本
    """
    
    # 计算各站点盈亏
    surplus = {s: current[s] - target[s] for s in stations}
    
    # 创建优化问题
    prob = LpProblem("Bike_Rebalancing", LpMinimize)
    
    # 变量：从 i 到 j 的移动量
    routes = [(i, j) for i in stations for j in stations if i != j]
    move = LpVariable.dicts("move", routes, lowBound=0, cat='Integer')
    
    # 目标：最小化总成本
    prob += lpSum([cost_matrix.get((i, j), 10) * move[(i, j)] for (i, j) in routes])
    
    # 平衡约束
    for s in stations:
        outflow = lpSum([move[(s, j)] for j in stations if s != j])
        inflow = lpSum([move[(i, s)] for i in stations if s != i])
        prob += outflow - inflow == surplus[s]
    
    # 求解
    prob.solve()
    
    # 提取结果
    movements = {}
    for (i, j) in routes:
        if move[(i, j)].varValue > 0:
            movements[(i, j)] = int(move[(i, j)].varValue)
    
    return movements, value(prob.objective)

# 定义移动成本（根据站点间距离）
cost_matrix = {
    ('A001', 'A002'): 2, ('A001', 'A003'): 2,
    ('A002', 'A001'): 2, ('A002', 'A003'): 3,
    ('A003', 'A001'): 2, ('A003', 'A002'): 3,
    ('B001', 'B002'): 2,
    ('B002', 'B001'): 2,
    ('C001', 'A001'): 5, ('C001', 'B001'): 4
}

# 优化
movements, total_cost = optimize_rebalancing(stations, current_bikes, target_bikes, cost_matrix)

print("\n车辆调度方案")
print("=" * 60)
for (i, j), amount in movements.items():
    print(f"{i} -> {j}: {amount} 辆")
print(f"\n总移动成本: {total_cost}")

In [ ]:
# 可视化调度网络
fig, ax = plt.subplots(figsize=(12, 8))

# 创建网络图
G = nx.DiGraph()

# 添加节点
for s in stations:
    G.add_node(s)

# 添加有调度的边
for (i, j), amount in movements.items():
    G.add_edge(i, j, weight=amount)

# 布局
pos = {
    'A001': (1, 2), 'A002': (2, 2), 'A003': (3, 2),
    'B001': (1, 1), 'B002': (2, 1),
    'C001': (2, 0)
}

# 节点颜色根据盈亏
surplus = {s: current_bikes[s] - target_bikes[s] for s in stations}
node_colors = ['green' if surplus[s] > 0 else 'red' if surplus[s] < 0 else 'gray' for s in stations]

# 绘制节点
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, ax=ax)
nx.draw_networkx_labels(G, pos, ax=ax)

# 绘制边
for (i, j) in G.edges():
    amount = G[i][j]['weight']
    nx.draw_networkx_edges(G, pos, edgelist=[(i, j)], 
                          width=amount/2, edge_color='blue',
                          arrowsize=20, ax=ax)
    # 标注数量
    mid_x = (pos[i][0] + pos[j][0]) / 2
    mid_y = (pos[i][1] + pos[j][1]) / 2
    ax.text(mid_x, mid_y, f"{amount}", fontsize=10, 
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='盈余站点'),
    Patch(facecolor='red', label='亏损站点'),
    Patch(facecolor='gray', label='平衡站点')
]
ax.legend(handles=legend_elements, loc='upper left')

ax.set_title('车辆调度网络\n绿色=盈余, 红色=亏损, 蓝色箭头=调度方向')
ax.axis('off')

plt.tight_layout()
plt.show()

## 6. 站点网络分析

In [ ]:
# 构建站点网络
station_network = nx.Graph()

# 添加站点
station_network.add_nodes_from(stations)

# 添加连接（根据地理位置）
connections = [
    ('A001', 'A002'), ('A001', 'B001'),
    ('A002', 'A003'), ('A002', 'B002'),
    ('A003', 'C001'),
    ('B001', 'B002'),
    ('B002', 'C001')
]

station_network.add_edges_from(connections)

# 计算中心性
degree_cent = nx.degree_centrality(station_network)
betweenness_cent = nx.betweenness_centrality(station_network)

print("站点网络中心性分析")
print("=" * 60)
for s in stations:
    print(f"{s}: 度中心性={degree_cent[s]:.3f}, 中介中心性={betweenness_cent[s]:.3f}")

In [ ]:
# 可视化网络
fig, ax = plt.subplots(figsize=(12, 8))

pos = {
    'A001': (1, 3), 'A002': (2, 3), 'A003': (3, 3),
    'B001': (1, 2), 'B002': (2, 2),
    'C001': (2, 1)
}

# 节点大小根据中心性
sizes = [1000 + 2000 * betweenness_cent[s] for s in station_network.nodes()]

nx.draw(station_network, pos, with_labels=True, 
        node_size=sizes, node_color='lightblue',
        font_size=12, ax=ax)

ax.set_title('站点网络拓扑\n节点大小=中介中心性')
ax.axis('off')

plt.tight_layout()
plt.show()

## 7. 决策支持系统

In [ ]:
# 综合决策支持函数
def decision_support(hour, is_weekend, temperature):
    """
    综合决策支持
    
    输入:
    - hour: 小时 (0-23)
    - is_weekend: 是否周末 (0/1)
    - temperature: 温度
    
    输出:
    - 预测需求
    - 调度建议
    - 关键站点
    """
    print(f"\n决策支持报告")
    print("=" * 60)
    print(f"时间: {hour}:00, {'周末' if is_weekend else '工作日'}, 温度: {temperature}°C")
    
    # 预测需求
    print("\n【需求预测】")
    for station in stations:
        features = pd.DataFrame([{
            'hour': hour,
            'is_weekend': is_weekend,
            'is_holiday': 0,
            'temperature': temperature,
            'is_rush_hour': 1 if (7 <= hour <= 9 or 17 <= hour <= 19) else 0,
            'is_night': 1 if hour <= 5 else 0,
            'station_encoded': station_encoder[station]
        }])
        
        pred_demand = best_model.predict(features)[0]
        print(f"  {station}: 预测需求 {pred_demand:.1f} 辆")
    
    # 调度建议
    print("\n【调度建议】")
    if 7 <= hour <= 9:
        print("  早高峰时段：建议向商业区站点调配车辆")
    elif 17 <= hour <= 19:
        print("  晚高峰时段：建议向住宅区站点调配车辆")
    elif 0 <= hour <= 5:
        print("  夜间时段：建议进行车辆维护和平衡调度")
    else:
        print("  平峰时段：维持现状或轻度调度")
    
    # 关键站点
    print("\n【关键站点】")
    key_stations = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:3]
    for s, cent in key_stations:
        print(f"  {s}: 中介中心性 {cent:.3f} (重要枢纽)")

# 示例
decision_support(hour=8, is_weekend=0, temperature=22)

## 8. 项目总结

### 8.1 主要成果

1. **需求预测**：R² > 0.8 的预测模型
2. **调度优化**：最小成本调度方案
3. **网络分析**：识别关键站点
4. **决策支持**：综合决策系统

### 8.2 技术栈总结

| 类别 | 工具/方法 |
|------|----------|
| 数据处理 | Pandas, NumPy |
| 可视化 | Matplotlib, Seaborn |
| 机器学习 | Scikit-learn |
| 优化建模 | PuLP |
| 网络分析 | NetworkX |

### 8.3 项目亮点

1. **ML + OR 结合**：预测 + 优化的完整流程
2. **端到端方案**：从数据到决策
3. **可解释性**：特征重要性、中心性分析
4. **实用价值**：可直接应用于实际运营

### 8.4 改进方向

1. **实时预测**：接入实时数据流
2. **动态调度**：考虑时间维度的优化
3. **多目标优化**：成本、服务水平、公平性
4. **用户行为**：考虑用户出行模式
5. **天气因素**：更精细的天气影响建模

## 9. 课程总结

### 10 周学习回顾

| 周次 | 主题 | 核心技能 |
|------|------|----------|
| Week 1 | 数据基础 | Pandas, NumPy, EDA |
| Week 2 | 概率模拟 | Monte Carlo, CLT |
| Week 3 | 统计推断 | Bootstrap, 置信区间 |
| Week 4 | ML 工作流 | 线性回归, 逻辑回归 |
| Week 5 | 模型比较 | 交叉验证, 超参数调优 |
| Week 6 | Mini Project | 完整项目流程 |
| Week 7 | 线性规划 | PuLP, 生产计划 |
| Week 8 | 整数规划 | 0-1 规划, 逻辑约束 |
| Week 9 | 网络优化 | NetworkX, 最短路, 最大流 |
| Week 10 | Capstone | 综合应用 |

### 核心能力

1. **数据思维**：观察 vs 干预，相关 vs 因果
2. **建模能力**：从问题到数学模型
3. **编程实践**：Python 生态系统
4. **系统思维**：ML + OR 的综合应用
5. **项目能力**：完整的端到端流程

### 后续学习建议

1. **深化 ML**：深度学习、时间序列、推荐系统
2. **拓展 OR**：动态规划、随机优化、博弈论
3. **工程实践**：MLOps、模型部署、实时系统
4. **领域应用**：物流、金融、医疗、能源
5. **研究前沿**：论文阅读、开源贡献